<a href="https://colab.research.google.com/github/souhirbenamor/EPF/blob/main/2025_Random_Forest_Bridging_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------------------------------------
# Step 0: Import Libraries and Set Random Seed
# -------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error  # Evaluation using MSE
import csv

# For reproducibility
np.random.seed(42)

# -------------------------------------------------------------
# Step 1: Data Loading and Preprocessing
# -------------------------------------------------------------
# Load the Excel file (adjust the file path as needed)
df = pd.read_excel('/content/EPF_data_Indiv.xlsx')

# Remove duplicate dates and convert the 'Date' column to datetime
df.drop(df.loc[df['Date'].duplicated()].index, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])

# Set the Date column as the index and drop the original 'Date' column
df.set_index(df['Date'], inplace=True)
df.drop(columns=['Date'], inplace=True)

# Filter out any observations before January 1, 2019
df = df[df.index >= '2019-01-01']

print("Data loaded and preprocessed (starting from 2019-01-01):")
print(df.head())


Data loaded and preprocessed (starting from 2019-01-01):
                         Price  Demand Day-ahead DE  \
Date                                                  
2019-01-01 00:59:59.983  -4.08             40553.00   
2019-01-01 01:59:59.983  -9.91             40261.25   
2019-01-01 02:59:59.983  -7.41             40603.25   
2019-01-01 03:59:59.983 -12.55             40904.25   
2019-01-01 04:59:59.983 -17.25             40783.50   

                         Wind and PV Day ahead (MWh/h)    Gas   Coal    CO2  
Date                                                                         
2019-01-01 00:59:59.983                       27384.00  21.98  75.44  24.73  
2019-01-01 01:59:59.983                       29010.25  21.98  75.44  24.73  
2019-01-01 02:59:59.983                       30359.25  21.98  75.44  24.73  
2019-01-01 03:59:59.983                       31409.25  21.98  75.44  24.73  
2019-01-01 04:59:59.983                       32934.75  21.98  75.44  24.73  


In [ ]:

# -------------------------------------------------------------
# Step 2: Split Data by Year
# -------------------------------------------------------------
# Get unique years in the dataset
unique_years = sorted(df.index.year.unique())
print("Unique years in data:", unique_years)

# Define training, validation, and test years.
# For example, if the data covers 2019-2024:
#   - Training: 2019, 2020, 2021
#   - Validation: 2022
#   - Test: 2023, 2024
train_years = [2019, 2020, 2021]
val_years = [2022]
test_years = [year for year in unique_years if year not in train_years + val_years]

print("Training years:", train_years)
print("Validation years:", val_years)
print("Test years:", test_years)

# Create DataFrames for each period
train_data = df[df.index.year.isin(train_years)]
val_data = df[df.index.year.isin(val_years)]
test_data = df[df.index.year.isin(test_years)]

# Combine training and validation data to form the initial history for training
history = pd.concat([train_data, val_data]).sort_index()
print("Combined training+validation shape:", history.shape)


Unique years in data: [2019, 2020, 2021, 2022, 2023, 2024]
Training years: [2019, 2020, 2021]
Validation years: [2022]
Test years: [2023, 2024]
Combined training+validation shape: (35064, 6)


In [ ]:

# -------------------------------------------------------------
# Step 3: Feature Selection for Training Data
# -------------------------------------------------------------
# Assume the target column is "price day ahead real" and use all other columns as features
target_column = 'Price'
features_list = [col for col in df.columns if col != target_column]

# Create training features and target variables from the history
X_train = history[features_list]
y_train = history[target_column]
print("Training features shape:", X_train.shape)
print("Training target shape:", y_train.shape)


Training features shape: (35064, 5)
Training target shape: (35064,)


In [ ]:

# -------------------------------------------------------------
# Step 4: Model Training and Hyperparameter Tuning
# -------------------------------------------------------------
# Define a parameter grid for GridSearchCV
param_grid = {
    'bootstrap': [True],
    'max_depth': [10, 250],
    'min_samples_leaf': [2, 5],
    'n_estimators': [100, 200]
}

# Initialize and run grid search
rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=2, verbose=2, n_jobs=1)
grid_search.fit(X_train, y_train)
print("Best GridSearch Score:", grid_search.best_score_)
print("Best Parameters:", grid_search.best_params_)

# Train the final model using chosen parameters (you may adjust these based on grid search)
rf_final = RandomForestRegressor(n_estimators=250, max_features=5, criterion='mae', max_depth=15, random_state=42)
rf_final.fit(X_train, y_train)
print("Final model trained.")

Fitting 2 folds for each of 8 candidates, totalling 16 fits
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, n_estimators=100; total time=   5.4s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, n_estimators=100; total time=   6.4s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, n_estimators=200; total time=  12.2s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, n_estimators=200; total time=  13.5s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=5, n_estimators=100; total time=   5.5s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=5, n_estimators=100; total time=   7.4s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=5, n_estimators=200; total time=  14.2s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=5, n_estimators=200; total time=  12.3s
[CV] END bootstrap=True, max_depth=250, min_samples_leaf=2, n_estimators=100; total time=   7.7s
[CV] END bootstrap=True, max_depth=250, min_samples_leaf=2, n_estimators=10

InvalidParameterError: The 'criterion' parameter of RandomForestRegressor must be a str among {'poisson', 'absolute_error', 'friedman_mse', 'squared_error'}. Got 'mae' instead.

In [ ]:

# -------------------------------------------------------------
# Step 5: Rolling Forecast on Test Data (Without Lagged Features)
# -------------------------------------------------------------
# In this section, the rolling forecast is performed on the test data.
# For each day in the test period, we:
#   1. Retrieve the 24 hourly rows for that day (using the features as they are in the Excel file).
#   2. Forecast the target variable ("price day ahead real") for those hours.
#   3. Append the forecasted results to the rolling history (simulating updated observations).
rolling_history = history.copy()  # Start with training + validation data
forecast_results = pd.DataFrame()   # DataFrame to store forecast results

# Get unique forecast days from the test data (based on dates)
test_days = pd.to_datetime(test_data.index.date).unique()
test_days = np.sort(test_days)
print("Number of forecast days in test period:", len(test_days))

for day in test_days:
    # Extract the test data for the current day (all hourly rows for that day)
    day_mask = test_data.index.normalize() == pd.Timestamp(day)
    X_forecast = test_data.loc[day_mask, features_list]

    # If no data for this day, skip forecasting
    if X_forecast.empty:
        print(f"No data available for day {day}. Skipping.")
        continue

    # Forecast the target variable for the current day
    y_forecast = rf_final.predict(X_forecast)

    # Create a DataFrame for the forecasted results
    df_forecast = pd.DataFrame({target_column: y_forecast}, index=X_forecast.index)

    # Append the forecasted results to rolling_history (simulating new observations)
    rolling_history = pd.concat([rolling_history, df_forecast])

    # Store the forecast results
    forecast_results = pd.concat([forecast_results, df_forecast])

print("Rolling forecast completed.")
print("Forecast results shape:", forecast_results.shape)


In [ ]:

# -------------------------------------------------------------
# Step 6: Evaluation and Visualization (Compute MSE)
# -------------------------------------------------------------
# Align the forecast results with the actual test data
forecast_results = forecast_results.sort_index()

# Extract the actual target values for the forecast period from test data
actual_test = test_data[[target_column]].loc[forecast_results.index]

# Combine actual and forecasted values into one DataFrame
results = actual_test.copy()
results['predicted_price'] = forecast_results[target_column]

# Compute the Mean Squared Error (MSE)
mse = mean_squared_error(results[target_column], results['predicted_price'])
print(f"Test MSE: {mse:.2f}")

# Plot actual vs. predicted prices
plt.figure(figsize=(28,9))
plt.xlabel('Date', fontsize=20)
plt.ylabel('Euro/mWh', fontsize=20)
plt.plot(results[target_column], label='Actual', color='r')
plt.plot(results['predicted_price'], label='Predicted', color='g')
plt.tick_params(axis='y', labelsize=20)
plt.legend(prop={'size': 20})
plt.title('Electricity Price Forecast vs Actual', fontsize=24)
plt.show()


In [ ]:

# -------------------------------------------------------------
# Step 7: Save the Forecast Results
# -------------------------------------------------------------
# Save the forecast results to CSV and Excel files
results.to_csv('Electricity_Price_Forecast_Results.csv', header=True, quoting=csv.QUOTE_NONE, escapechar=' ')
results.to_excel('Electricity_Price_Forecast_Results.xlsx', header=True)
print("Forecast results saved to CSV and Excel.")
